In [1]:
import os
os.environ["OPEN3D_DISABLE_WEB_VISUALIZER"] = "true"

import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import json
import imageio.v3
import imageio.v2 as imageio
import mvlm
import copy

from matplotlib import cm
import tempfile

In [2]:
def load_landmarks(path):
    path = Path(path)

    if path.suffix.lower() == ".npy":
        L = np.load(path)
        return np.asarray(L, dtype=np.float64)

    if path.suffix.lower() == ".txt":
        try:
            L = np.loadtxt(path, delimiter=",")
        except ValueError:
            L = np.loadtxt(path)
        L = np.asarray(L, dtype=np.float64)
        if L.ndim != 2 or L.shape[1] != 3:
            raise ValueError(f"TXT file must have shape (K,3), got {L.shape}")
        return L

    if path.suffix.lower() == ".json":
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        if isinstance(data, dict) and "markups" in data:
            cps = data["markups"][0]["controlPoints"]
            return np.array([cp["position"] for cp in cps], dtype=np.float64)

        if isinstance(data, list):
            return np.array(data, dtype=np.float64)

        if isinstance(data, dict) and "landmarks" in data:
            return np.array(data["landmarks"], dtype=np.float64)

    raise ValueError(f"Unsupported landmark format: {path}")

In [3]:
def per_landmark_3d_error(landmarks_true, landmarks_pred, one_based_index=True):
    """
    Per-landmark Euclidean distance between true and predicted 3D landmarks.
    """
    A = np.asarray(landmarks_true, dtype=np.float64)
    B = np.asarray(landmarks_pred, dtype=np.float64)

    if A.shape != B.shape or A.ndim != 2 or A.shape[1] != 3:
        raise ValueError(f"Expected both arrays to have shape (K,3), got {A.shape} and {B.shape}")

    diff = B - A
    dist = np.linalg.norm(diff, axis=1)

    idx = np.arange(1, len(dist) + 1) if one_based_index else np.arange(len(dist))

    df = pd.DataFrame({
        "landmark": idx,
        "dx": diff[:, 0],
        "dy": diff[:, 1],
        "dz": diff[:, 2],
        "dist_3d": dist,
    })

    return df


def summarize_3d_error(df, col="dist_3d"):
    vals = df[col].to_numpy()
    return {
        "mean": float(np.mean(vals)),
        "median": float(np.median(vals)),
        "std": float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0,
        "min": float(np.min(vals)),
        "max": float(np.max(vals)),
    }

In [4]:
def _copy_mesh(mesh: o3d.geometry.TriangleMesh) -> o3d.geometry.TriangleMesh:
    
    m = o3d.geometry.TriangleMesh(mesh)
    m.vertices = o3d.utility.Vector3dVector(np.asarray(mesh.vertices).copy())
    m.triangles = o3d.utility.Vector3iVector(np.asarray(mesh.triangles).copy())
    if mesh.has_vertex_normals():
        m.vertex_normals = o3d.utility.Vector3dVector(np.asarray(mesh.vertex_normals).copy())
    if mesh.has_triangle_normals():
        m.triangle_normals = o3d.utility.Vector3dVector(np.asarray(mesh.triangle_normals).copy())
    return m

In [5]:
def pointcloud_as_spheres(pcd, radius=0.5, color=[0, 0, 1]):
    spheres = []
    for p in np.asarray(pcd.points):
        sphere = o3d.geometry.TriangleMesh.create_sphere(radius=radius)
        sphere.compute_vertex_normals()
        sphere.translate(p)
        sphere.paint_uniform_color(color)
        spheres.append(sphere)
    return spheres

In [6]:
def indexed_colors(K, cmap_name="tab20"):
    cmap = plt.get_cmap(cmap_name, K if K <= 20 else 20)
    return np.array([cmap(i % cmap.N)[:3] for i in range(K)], dtype=np.float64)


#def spheres_from_landmarks(landmarks, colors, radius=0.3, resolution=16):
#    geoms = []
#    for p, c in zip(np.asarray(landmarks), np.asarray(colors)):
#        sph = o3d.geometry.TriangleMesh.create_sphere(radius=radius, resolution=resolution)
#        sph.compute_vertex_normals()
#        sph.translate(p)
#        sph.paint_uniform_color(np.asarray(c).tolist())
#        geoms.append(sph)
#    return geoms


def marker_mesh(kind="sphere", radius=0.3, resolution=16):
    if kind == "sphere":
        m = o3d.geometry.TriangleMesh.create_sphere(radius=radius, resolution=resolution)
    elif kind == "tetra":
        m = o3d.geometry.TriangleMesh.create_tetrahedron(radius=radius)
    elif kind == "octa":
        m = o3d.geometry.TriangleMesh.create_octahedron(radius=radius)
    else:
        raise ValueError(f"Unknown marker kind: {kind}")

    m.compute_vertex_normals()
    return m


def markers_from_landmarks(landmarks, colors, kind="sphere", radius=0.3, resolution=16):
    geoms = []
    for p, c in zip(np.asarray(landmarks), np.asarray(colors)):
        g = marker_mesh(kind=kind, radius=radius, resolution=resolution)
        g.translate(p)
        g.paint_uniform_color(np.asarray(c).tolist())
        geoms.append(g)
    return geoms


def colored_lines_between(A, B, colors):
    A = np.asarray(A, dtype=np.float64)
    B = np.asarray(B, dtype=np.float64)
    K = A.shape[0]

    points = np.vstack([A, B])
    lines = [[i, i + K] for i in range(K)]

    ls = o3d.geometry.LineSet()
    ls.points = o3d.utility.Vector3dVector(points)
    ls.lines = o3d.utility.Vector2iVector(lines)
    ls.colors = o3d.utility.Vector3dVector(np.asarray(colors, dtype=np.float64))
    return ls

In [7]:
def compute_scene_bbox(geoms):
    mins, maxs = [], []
    for g in geoms:
        bbox = g.get_axis_aligned_bounding_box()
        mins.append(bbox.min_bound)
        maxs.append(bbox.max_bound)

    mins = np.vstack(mins)
    maxs = np.vstack(maxs)
    scene_min = mins.min(axis=0)
    scene_max = maxs.max(axis=0)
    center = 0.5 * (scene_min + scene_max)
    extent = scene_max - scene_min
    radius = np.linalg.norm(extent) * 1.5
    return center, radius


def show_translucent_scene_with_safe_gif(
    mesh,
    landmarks_true,
    landmarks_pred,
    cols,
    out_path="output/landmarks_transparent.gif",
    sphere_radius=0.3,
    mesh_alpha=0.18,
    n_frames=60,
    fps=10,
    width=1200,
    height=800,
    elevation=0.3,
    fov=60.0,
):
    mesh_vis = copy.deepcopy(mesh)
    mesh_vis.compute_vertex_normals()

    true_markers = markers_from_landmarks(
        landmarks_true, cols,
        kind="sphere",
        radius=sphere_radius,
        resolution=16
    )
    
    pred_cols = np.clip(np.asarray(cols) * 0.6, 0, 1)
    pred_markers = markers_from_landmarks(
        landmarks_pred, pred_cols,
        kind="tetra",
        radius=1.0 #sphere_radius
    )
    ls = colored_lines_between(landmarks_true, landmarks_pred, cols)

    geoms_for_bbox = [mesh_vis, ls] + true_markers + pred_markers
    center, radius = compute_scene_bbox(geoms_for_bbox)
    gif_zoom_factor = 0.55   # smaller = closer
    gif_radius = radius * gif_zoom_factor

    mesh_mat = o3d.visualization.rendering.MaterialRecord()
    mesh_mat.shader = "defaultLitTransparency"
    mesh_mat.base_color = [0.85, 0.85, 0.85, mesh_alpha]

    sphere_mat = o3d.visualization.rendering.MaterialRecord()
    sphere_mat.shader = "defaultLit"

    line_mat = o3d.visualization.rendering.MaterialRecord()
    line_mat.shader = "unlitLine"
    line_mat.line_width = 8.0

    draw_list = [
        {"name": "mesh", "geometry": mesh_vis, "material": mesh_mat},
        {"name": "lines", "geometry": ls, "material": line_mat},
    ]

    for i, g in enumerate(true_markers):
        draw_list.append({
            "name": f"true_{i}",
            "geometry": g,
            "material": sphere_mat
        })

    for i, g in enumerate(pred_markers):
        draw_list.append({
            "name": f"pred_{i}",
            "geometry": g,
            "material": sphere_mat
        })

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    state = {
        "recording": False,
        "frame_idx": 0,
        "tmpdir": None,
        "frame_paths": [],
    }

    def start_gif(o3dvis):
        if state["recording"]:
            print("Already recording.")
            return
    
        state["recording"] = True
        state["frame_idx"] = 0
        state["frame_paths"] = []
        state["tmpdir"] = Path(tempfile.mkdtemp(prefix="o3d_gif_"))
        print(f"Recording {n_frames} frames...")
    
        o3dvis.animation_frame_delay = 0.01  # optional, smaller = faster
    
        def on_frame(vis, current_time):
            i = state["frame_idx"]
    
            if i >= n_frames:
                imgs = [imageio.imread(p) for p in state["frame_paths"]]
                imageio.mimsave(str(out_path), imgs, fps=fps, loop=0)
                print(f"Saved GIF: {out_path}")
    
                state["recording"] = False
                vis.is_animating = False
                vis.set_on_animation_frame(None)
                return
    
            angle = 2.0 * np.pi * i / n_frames
            eye = np.array([
                center[0] + gif_radius * np.cos(angle),
                center[1] + gif_radius * np.sin(angle),
                center[2] + gif_radius * elevation
            ], dtype=np.float32)
            
            vis.setup_camera(
                fov,
                center.astype(np.float32),
                eye,
                np.array([0.0, 0.0, 1.0], dtype=np.float32)
            )
            vis.post_redraw()
    
            frame_path = state["tmpdir"] / f"frame_{i:03d}.png"
            vis.export_current_image(str(frame_path))
            state["frame_paths"].append(frame_path)
            state["frame_idx"] += 1
    
        o3dvis.set_on_animation_frame(on_frame)
        o3dvis.is_animating = True

    start_eye = np.array([
        center[0] + radius,
        center[1],
        center[2] + radius * elevation
    ], dtype=np.float32)
    
    start_up = np.array([0.0, 0.0, 1.0], dtype=np.float32)
    start_center = center.astype(np.float32)
    
    o3d.visualization.draw(
        draw_list,
        title="Translucent mesh + landmarks",
        width=width,
        height=height,
        bg_color=(1.0, 1.0, 1.0, 1.0),
        show_ui=True,
        show_skybox=False,
        lookat=start_center,
        eye=start_eye,
        up=start_up,
        field_of_view=fov,
        actions=[("Save GIF", start_gif)]
    )

In [8]:
def generate_gif(meshes, colors, out_path):
    n_frames = 60
    width, height = 800, 600

    vis = o3d.visualization.Visualizer()
    vis.create_window(visible=True, width=width, height=height)
    for idx, geom in enumerate(meshes):
        if isinstance(geom, o3d.geometry.TriangleMesh):
            geom = _copy_mesh(geom)
            geom.compute_vertex_normals()

            has_tex = hasattr(geom, "textures") and len(geom.textures) > 0 and geom.triangle_uvs is not None and len(geom.triangle_uvs) > 0
            if not has_tex:
                geom.paint_uniform_color(colors[idx])
            
            vis.add_geometry(geom)

        elif isinstance(geom, o3d.geometry.PointCloud):
            pcd = o3d.geometry.PointCloud(geom)  # copy
            pts = np.asarray(pcd.points)

            sphere_radius = 0.8 
            sphere_res = 12      

            for p in pts:
                sph = o3d.geometry.TriangleMesh.create_sphere(
                    radius=sphere_radius, resolution=sphere_res
                )
                sph.compute_vertex_normals()
                sph.translate(p)
                sph.paint_uniform_color(colors[idx])
                vis.add_geometry(sph)
        else:
            raise TypeError(f"Unsupported geometry type: {type(geom)}")

    opt = vis.get_render_option()
    opt.background_color = np.array([5.1, 6.7, 9])
    opt.point_size = 10.0
    
    ctr = vis.get_view_control()
    bbox = geom.get_axis_aligned_bounding_box()
    center = bbox.get_center()
    
    radius = np.linalg.norm(bbox.get_extent()) * 1.2 # 1.5
    
    frames = []
    for i in range(n_frames):
        angle = 2 * np.pi * i / n_frames
    
        # --- CAMERA PATH ---
        # orbit around z-axis with slight elevation
        cam_x = center[0] + radius * np.cos(angle)
        cam_y = center[1] + radius * np.sin(angle)
        cam_z = center[2] + radius * 0.3
    
        ctr.set_lookat(center)
        ctr.set_front([center[0] - cam_x,
                       center[1] - cam_y,
                       center[2] - cam_z])
        ctr.set_up([0, 0, 1])
        ctr.set_zoom(0.8)
    
        vis.poll_events()
        vis.update_renderer()
    
        img = vis.capture_screen_float_buffer(False)
        img = (np.asarray(img) * 255).astype(np.uint8)
        frames.append(img)
    
    vis.destroy_window()
    # save GIF
    imageio.mimsave(out_path, frames, fps=10, loop=0)
    
    print(f"Saved rotating GIF as: {out_path}")

In [9]:
#ObjectToVis = Path("data/UNSEEN/SAPM-MA-03201")
ObjectToVis = Path("data/UNSEEN-GAZELLE/SAPM-MA-02067")

file = Path(f"{ObjectToVis}.obj")
#lm_true = Path(f"{ObjectToVis}_geomedian.mrk.json")
lm_true = Path(f"{ObjectToVis}.mrk.json")
lm_pred = Path(f"{ObjectToVis}_predicted.txt")

pathToOut = Path("output/")

mesh = o3d.io.read_triangle_mesh(file, enable_post_processing=True)
landmarks_true = load_landmarks(lm_true)
landmarks_pred = load_landmarks(lm_pred)

In [10]:
K = landmarks_true.shape[0]
cols = indexed_colors(K, "Paired") #tab20
mesh = o3d.io.read_triangle_mesh(file, enable_post_processing=True)

show_translucent_scene_with_safe_gif(
    mesh,
    landmarks_true,
    landmarks_pred,
    cols,
    out_path="output/landmarks_02067.gif",
    sphere_radius=0.8,
    mesh_alpha=0.5,
    fov=25.0,
    n_frames=60,
    fps=10
)

Recording 60 frames...
Saved GIF: output\landmarks_02067.gif
Saved GIF: output\landmarks_02067.gif


In [11]:
df_model = per_landmark_3d_error(landmarks_true, landmarks_pred)
display(df_model)

model_summary = summarize_3d_error(df_model)
print(model_summary)

,landmark,dx,dy,dz,dist_3d
0,1,-0.833419,-0.868382,-0.492128,1.300333
1,2,0.283725,0.504536,0.320090,0.661449
2,3,-0.386464,-0.041782,-0.426236,0.576869
3,4,0.046122,0.093257,0.203149,0.228240
4,5,0.337393,0.598404,0.254729,0.732672
5,6,0.406912,-0.190620,0.187903,0.487054
6,7,-0.116204,-0.363548,-0.444944,0.586213
7,8,1.149654,-1.231356,-0.845258,1.884782
8,9,0.341932,2.069466,2.749814,3.458480
9,10,-0.486780,-0.487697,0.239977,0.729652


{'mean': 1.1932602929558453, 'median': 0.7326715223878993, 'std': 0.8945130732910622, 'min': 0.22824025720020072, 'max': 3.4584796548033077}


In [12]:
# Calculate Human Error

In [13]:
def apply_similarity_transform(P, R, t, s=1.0):
    """
    Apply similarity transform to Nx3 points:
        P_aligned = s * (P @ R.T) + t
    """
    P = np.asarray(P, dtype=np.float64)
    return s * (P @ R.T) + t


def fit_kabsch(P, Q, allow_scaling=False):
    """
    Fit rigid (or similarity) transform mapping P -> Q.
    Returns R, t, s such that:
        Q ≈ s * (P @ R.T) + t
    """
    P = np.asarray(P, dtype=np.float64)
    Q = np.asarray(Q, dtype=np.float64)

    if P.shape != Q.shape or P.ndim != 2 or P.shape[1] != 3:
        raise ValueError(f"Expected P and Q with shape (K,3), got {P.shape} and {Q.shape}")

    cP = P.mean(axis=0)
    cQ = Q.mean(axis=0)

    X = P - cP
    Y = Q - cQ

    H = X.T @ Y
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T

    # avoid reflection
    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T

    s = 1.0
    if allow_scaling:
        denom = np.sum(X ** 2)
        s = np.sum(S) / denom if denom > 0 else 1.0

    t = cQ - s * (cP @ R.T)
    return R, t, s


def generalized_procrustes(landmark_sets, n_iter=20, tol=1e-8, allow_scaling=False):
    """
    Generalized Procrustes alignment for repeated annotations of the same specimen.
    Returns:
        aligned_sets : list of aligned (K,3) arrays
        consensus    : mean landmark configuration after alignment
        transforms   : list of (R, t, s)
    """
    landmark_sets = [np.asarray(L, dtype=np.float64) for L in landmark_sets]

    if len(landmark_sets) < 2:
        raise ValueError("Need at least two landmark sets for alignment.")

    K = landmark_sets[0].shape
    if any(L.shape != K for L in landmark_sets):
        raise ValueError("All landmark sets must have the same shape and landmark order.")

    consensus = landmark_sets[0].copy()
    prev_change = None

    for _ in range(n_iter):
        aligned_sets = []
        transforms = []

        for L in landmark_sets:
            R, t, s = fit_kabsch(L, consensus, allow_scaling=allow_scaling)
            L_aligned = apply_similarity_transform(L, R, t, s)
            aligned_sets.append(L_aligned)
            transforms.append((R, t, s))

        new_consensus = np.mean(np.stack(aligned_sets, axis=0), axis=0)
        change = np.linalg.norm(new_consensus - consensus)

        consensus = new_consensus

        if prev_change is not None and abs(prev_change - change) < tol:
            break
        prev_change = change

    return aligned_sets, consensus, transforms


def pairwise_landmark_distances(aligned_sets):
    """
    Pairwise per-landmark distances across all aligned manual annotations.
    Returns array of shape (n_pairs, K).
    """
    arr = np.stack(aligned_sets, axis=0)  # (N,K,3)
    N, K, _ = arr.shape

    dists = []
    for i in range(N):
        for j in range(i + 1, N):
            d = np.linalg.norm(arr[i] - arr[j], axis=1)  # (K,)
            dists.append(d)

    return np.stack(dists, axis=0)  # (n_pairs, K)


def human_variability_report(manual_landmark_sets, one_based_index=True, allow_scaling=False):
    """
    Align repeated manual annotations of the same specimen and compute:
      - mean distance to consensus per landmark
      - std distance to consensus per landmark
      - mean pairwise distance per landmark

    Returns:
      df_human, aligned_sets, consensus, d_to_consensus, pairwise
    """
    aligned_sets, consensus, transforms = generalized_procrustes(
        manual_landmark_sets,
        allow_scaling=allow_scaling
    )

    arr = np.stack(aligned_sets, axis=0)                 # (N,K,3)
    d_to_consensus = np.linalg.norm(arr - consensus[None, :, :], axis=2)  # (N,K)
    pairwise = pairwise_landmark_distances(aligned_sets)                   # (P,K)

    K = consensus.shape[0]
    idx = np.arange(1, K + 1) if one_based_index else np.arange(K)

    df_human = pd.DataFrame({
        "landmark": idx,
        "human_mean_to_consensus": d_to_consensus.mean(axis=0),
        "human_std_to_consensus": d_to_consensus.std(axis=0, ddof=1) if arr.shape[0] > 1 else np.zeros(K),
        "human_max_to_consensus": d_to_consensus.max(axis=0),
        "human_mean_pairwise": pairwise.mean(axis=0),
        "human_median_pairwise": np.median(pairwise, axis=0),
    })

    return df_human, aligned_sets, consensus, d_to_consensus, pairwise

In [14]:
manual_paths = [
    Path("data/UNSEEN/SAPM-MA-03201.mrk.json"),
    Path("data/UNSEEN/SAPM-MA-03211.mrk.json"),
    Path("data/UNSEEN/SPM-MA-03227.mrk.json"),
    Path("data/UNSEEN/THE-MA-49.mrk.json"),
    Path("data/UNSEEN/THE-MA-60.mrk.json"),
]

manual_sets = [load_landmarks(p) for p in manual_paths]

df_human, aligned_manual_sets, manual_consensus, d_to_consensus, pairwise = human_variability_report(
    manual_sets,
    allow_scaling=False  # keep False if it is really the same specimen, just rotated/translated
)

display(df_human)
print({
    "mean_human_to_consensus": float(df_human["human_mean_to_consensus"].mean()),
    "mean_human_pairwise": float(df_human["human_mean_pairwise"].mean())
})

,landmark,human_mean_to_consensus,human_std_to_consensus,human_max_to_consensus,human_mean_pairwise,human_median_pairwise
0,1,2.037305,0.846018,2.855392,3.115821,3.099131
1,2,2.041234,0.524560,2.673711,3.156015,3.108055
2,3,1.688098,1.025460,3.085579,2.764838,2.527383
3,4,1.729160,0.841765,2.888206,2.737084,2.489339
4,5,1.542825,0.693464,2.635031,2.296391,2.271792
5,6,2.062898,0.645857,2.973210,3.188921,3.342022
6,7,1.894855,0.776176,2.505925,2.985809,2.604196
7,8,1.930339,0.945038,2.811735,3.113648,2.666290
8,9,1.559320,0.865568,2.926075,2.476208,2.371645
9,10,1.903764,0.705873,2.908976,2.968234,2.732112


{'mean_human_to_consensus': 1.8450146538972982, 'mean_human_pairwise': 2.878364291541594}
